# 07 – Cost-based threshold analysis (business view of errors)
A missed at-risk student (false negative, FN) and an unnecessary advising contact (false positive, FP)
do not cost the same. The real costs are NOT in the dataset, so we express the cost of one FP as 1 unit
and test several ILLUSTRATIVE FN:FP cost ratios. The institution should replace these with its own figures.

In [ ]:
import sys
from pathlib import Path
# make the shared module in ../src importable (no personal paths)
sys.path.append(str(Path.cwd().parent / 'src'))

In [ ]:
import numpy as np
import pandas as pd
from common import *

pred = pd.read_csv(RES_DIR / "test_predictions.csv")
y = pred["actual_status"].isin(AT_RISK).values
rows = []
for t in np.arange(0.10, 0.91, 0.05):
    flag = pred["risk_score"].values >= t
    fn, fp = int((y & ~flag).sum()), int((~y & flag).sum())
    row = {"threshold": round(t, 2), "flagged_pct": flag.mean() * 100, "FN_missed": fn, "FP_unneeded": fp,
           "recall": (y & flag).sum() / y.sum(), "precision": (y & flag).sum() / max(flag.sum(), 1)}
    for r in (1, 3, 5, 10):
        row[f"cost_ratio_{r}"] = (r * fn + fp) / len(y) * 1000      # cost units per 1,000 students
    rows.append(row)
df = pd.DataFrame(rows).round(3)
df.to_csv(RES_DIR / "cost_by_threshold.csv", index=False)
best = pd.DataFrame([{"fn_to_fp_cost_ratio": r, "best_threshold": df.loc[df[f"cost_ratio_{r}"].idxmin(), "threshold"],
                      "min_cost_per_1000": df[f"cost_ratio_{r}"].min(),
                      "cost_at_0.5": df.loc[df.threshold == 0.5, f"cost_ratio_{r}"].iloc[0],
                      "no_model_cost": r * y.mean() * 1000} for r in (1, 3, 5, 10)]).round(2)
best.to_csv(RES_DIR / "best_threshold_by_cost_ratio.csv", index=False)
print(df[["threshold", "flagged_pct", "FN_missed", "FP_unneeded", "recall", "precision"]].to_string()); print(best)